# 06 - Get Structures (Apo/Holo)

Notebook for retrieving/predicting protein structures using selectable tools (RCSB search, Boltz-2 via OpenProtein, Boltz-2 local).

## Python Path Setup
Ensure project-root imports work whether Jupyter starts from repo root or `notebooks/`.

In [1]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
repo_root = cwd.parent if cwd.name == "notebooks" else cwd
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
src_root = repo_root / "src"
if src_root.exists() and str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

## Imports
Load OpenProtein Boltz-2 predictor and shared path helpers.

In [3]:
import json
from abcode.core.paths import resolve_input_path
from project_config.variables import address_dict, subfolders
from abcode.steps.get_structures_apo_holo import run_structure_prediction
from abcode.tools.utils.struct_utils import convert_cif_to_pdb_pymol, visualize_structures

## User Inputs
Specify known PDB id (optional), sequence(s), ligand(s), and preferred structure tool.

In [4]:
root_key = "biostream-developability-data"

user_inputs = {
    "pdb_id": "",  # optional known structure id
    "struct_name": "Fv_000",
    "sequences": [       "QVKLQESGAELARPGASVKLSCKASGYTFTNYWMQWVKQRPGQGLDWIGAIYPGDGNTRYTHKFKGKATLTADKSSSTAYMQLSSLASEDSGVYYCARGEGNYAWFAYWGQGTTVTVSS",
    "DIELTQSPASLSASVGETVTITCQASENIYSYLAWHQQKQGKSPQLLVYNAKTLAGGVSSRFSGSGSGTHFSLKIKSLQPEDFGIYYCQHHYGILPTFGGGTKLEIK"
    ],
    "ligands": [],
    "preferred_structure_tool": "boltz2_openprotein",  # rcsb_search | boltz2_openprotein | boltz2_local
    "predict_affinity": False,
}

user_inputs

{'pdb_id': '',
 'struct_name': 'Fv_000',
 'sequences': ['QVKLQESGAELARPGASVKLSCKASGYTFTNYWMQWVKQRPGQGLDWIGAIYPGDGNTRYTHKFKGKATLTADKSSSTAYMQLSSLASEDSGVYYCARGEGNYAWFAYWGQGTTVTVSS',
  'DIELTQSPASLSASVGETVTITCQASENIYSYLAWHQQKQGKSPQLLVYNAKTLAGGVSSRFSGSGSGTHFSLKIKSLQPEDFGIYYCQHHYGILPTFGGGTKLEIK'],
 'ligands': [],
 'preferred_structure_tool': 'boltz2_openprotein',
 'predict_affinity': False}

## Resolve Output Paths
Set output paths under selected data root (`pdb/` subfolder).

In [5]:
project_root = repo_root
data_root = (project_root / address_dict[root_key]).resolve()
pdb_dir = (data_root / subfolders["pdb"]).resolve()
pdb_dir.mkdir(parents=True, exist_ok=True)

out_cif = pdb_dir / f"{user_inputs['struct_name']}_boltz2.cif"
out_summary = pdb_dir / f"{user_inputs['struct_name']}_boltz2_summary.json"

pdb_dir, out_cif, out_summary

(PosixPath('/Users/charmainechia/Documents/projects/biostream-developability-data/pdb'),
 PosixPath('/Users/charmainechia/Documents/projects/biostream-developability-data/pdb/Fv_000_boltz2.cif'),
 PosixPath('/Users/charmainechia/Documents/projects/biostream-developability-data/pdb/Fv_000_boltz2_summary.json'))

## Run Prediction
Run selected tool/API and return prediction outputs (Boltz-2 OpenProtein path implemented).

In [6]:
result = run_structure_prediction(user_inputs, out_cif, out_summary)
result

Session: <openprotein.OpenProtein object at 0x134164f10>
msa: 91ece9ba-13e5-4d2a-bff7-cf6543d11f01


Waiting:   0%|          | 0/100 [00:00<?, ?it/s]Error processing line 1 of /Users/charmainechia/miniconda3/envs/agentic-protein-design/lib/python3.11/site-packages/distutils-precedence.pth:

  Traceback (most recent call last):
    File "<frozen site>", line 195, in addpackage
    File "<string>", line 1, in <module>
  ModuleNotFoundError: No module named '_distutils_hack'

Remainder of file ignored
Waiting: 100%|██████████| 100/100 [17:07<00:00, 10.28s/it, status=SUCCESS]


{'job_id': '6ce483b0-e6f1-47e8-93ef-9ab69b13fff8',
 'protein_chains': ['A', 'B'],
 'ligand_chains': [],
 'predict_affinity': False,
 'binder_chain': None,
 'status': 'JobStatus.SUCCESS',
 'cif_path': '/Users/charmainechia/Documents/projects/biostream-developability-data/pdb/Fv_000_boltz2.cif',
 'plddt_shape': [1, 226],
 'pae_shape': [1, 226, 226],
 'pde_shape': [1, 226, 226],
 'confidence': {'confidence_score': 0.9724475145339966,
  'ptm': 0.9590503573417664,
  'iptm': 0.9394851326942444,
  'ligand_iptm': 0.0,
  'protein_iptm': 0.9394851326942444,
  'complex_plddt': 0.9806880354881287,
  'complex_iplddt': 0.9783422350883484,
  'complex_pde': 0.33470267057418823,
  'complex_ipde': 0.5738016963005066,
  'chains_ptm': {'0': 0.975214421749115, '1': 0.9811409711837769},
  'pair_chains_iptm': {'0': {'0': 0.975214421749115, '1': 0.939187228679657},
   '1': {'0': 0.9394851326942444, '1': 0.9811409711837769}}},
 'tool': 'boltz2_openprotein',
 'implemented': True}

## Inspect Saved Outputs
Display saved summary content and output artifact paths.

In [6]:
if out_summary.exists():
    print(out_summary)
    print(json.dumps(json.loads(out_summary.read_text(encoding="utf-8")), indent=2)[:4000])
else:
    print("No summary file written yet.")

print("CIF exists:", out_cif.exists(), out_cif)

/Users/charmainechia/Documents/projects/PIPS/PIPS2-UPOs-data/pdb/ET096_S82_boltz2_summary.json
{
  "job_id": "55558cd5-d0ca-4879-8340-22e2cab1e267",
  "protein_chains": [
    "A"
  ],
  "ligand_chains": [
    "B",
    "C",
    "D"
  ],
  "predict_affinity": false,
  "binder_chain": "B",
  "status": "JobStatus.SUCCESS",
  "cif_path": "/Users/charmainechia/Documents/projects/PIPS/PIPS2-UPOs-data/pdb/ET096_S82_boltz2.cif",
  "plddt_shape": [
    1,
    298
  ],
  "pae_shape": [
    1,
    298,
    298
  ],
  "pde_shape": [
    1,
    298,
    298
  ],
  "confidence": {
    "confidence_score": 0.9383544325828552,
    "ptm": 0.9397873878479004,
    "iptm": 0.9836310148239136,
    "ligand_iptm": 0.9836310148239136,
    "protein_iptm": 0.0,
    "complex_plddt": 0.9270352125167847,
    "complex_iplddt": 0.9457175731658936,
    "complex_pde": 0.3578200340270996,
    "complex_ipde": 0.4441279172897339,
    "chains_ptm": {
      "0": 0.9305444359779358,
      "1": 0.9187964797019958,
      "2": 0

In [7]:
# 1. Open CIF and Save as PDB using PyMOL
out_pdb = convert_cif_to_pdb_pymol(out_cif, str(out_cif).replace('.cif', '.pdb'))

# Visualize PDB in NGLView
view = visualize_structures([out_pdb], show_res_near_ligand=6, protein_chain_id='A', ligand_chain_id='D')
view

0 /Users/charmainechia/Documents/projects/PIPS/PIPS2-UPOs-data/pdb/ET096_S82_boltz2.pdb


NGLWidget()